# Gnani Prisma (ASR) & Timbre (TTS) Evaluation

**Engineer:** Ashish Vipin  
**Role:** AI Solutions Engineer Evaluation  
**Target Systems:** Gnani Prisma v3.0 (ASR) & Timbre v2.5 (TTS) via `api.vachana.ai`

---

## Overview
This notebook runs and inspects the end-to-end evaluation harness:
1. **Part 1 — Prisma ASR**: Evaluates **both** a 22-clip **Real-Human Speech Suite** (AI4Bharat Svarah Indian English, Google FLEURS Hindi, OpenSLR Telugu/Tamil/Bengali, plus real-speech babble noise & 8 kHz μ-law telephony) and a 30-clip **Synthetic Smoke Suite** (Timbre-synthesized controlled cases), reporting **both verbatim and format-normalised WER/CER** implemented from scratch via Wagner-Fischer DP.
2. **Part 2 — Timbre TTS**: Evaluates 29 sentences across 9 operational categories (3 repeats each) using **component metrics**—p50/p95 full-response HTTP latency, RTF, retry counts, 25 ms frame-level RMS silence ratio, clipping, and round-trip Prisma ASR WER (plus optional `faster-whisper` cross-check on Colab).

In [1]:
import os
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from config import GNANI_API_KEY, ASR_MODEL, TTS_MODEL
from src.metrics import compute_wer, compute_cer
from src.gnani_client import GnaniClient

print('API Key Configured:', 'Yes' if GNANI_API_KEY else 'No')
print(f'Target Models: ASR={ASR_MODEL}, TTS={TTS_MODEL}')

API Key Configured: Yes
Target Models: ASR=prisma-v3.0, TTS=timbre-v2.5


/Users/mac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Optional: Re-Run Live Evaluation Directly From This Notebook
Set `RUN_LIVE_EVAL = True` to re-run both Part 1 (Prisma ASR) and Part 2 (Timbre TTS) live against `api.vachana.ai`. When `False`, subsequent cells inspect the latest saved empirical results.

In [2]:
RUN_LIVE_EVAL = False
TTS_REPEATS = 3
USE_INDEPENDENT_WHISPER = False  # Set True on Colab after `!pip install faster-whisper`

if RUN_LIVE_EVAL:
    from src.asr_evaluator import run_asr_suites
    from src.tts_evaluator import run_tts_evaluation

    run_asr_suites(
        suites={
            'real_human': 'data/asr/real_manifest.json',
            'synthetic_smoke': 'data/asr/test_manifest.json',
        },
        output_path='results/asr_results.json',
    )
    run_tts_evaluation(
        test_set_path='data/tts/test_sentences.json',
        output_dir='data/tts/output',
        output_path='results/tts_results.json',
        repeats=TTS_REPEATS,
        use_independent_asr=USE_INDEPENDENT_WHISPER,
    )
else:
    print('RUN_LIVE_EVAL=False — loading saved empirical runs from results/*.json')

RUN_LIVE_EVAL=False — loading saved empirical runs from results/*.json


## Part 1: Scratch WER/CER Metric Verification (Wagner-Fischer DP)

In [3]:
sample_ref = 'the cat sat on the mat'
sample_hyp = 'the cat sit on a mat'

res_wer = compute_wer(sample_ref, sample_hyp)
res_cer = compute_cer(sample_ref, sample_hyp)
hi_wer = compute_wer('कृपया मेरा आधार नंबर से लिंक करें', 'कृपया मेरा आधार नंबर से लिंक करें।', language='hi')

print('Reference :', sample_ref)
print('Hypothesis:', sample_hyp)
print(f"WER: {res_wer['wer']:.2%} (S={res_wer['substitutions']}, D={res_wer['deletions']}, I={res_wer['insertions']})")
print(f"CER: {res_cer['cer']:.2%}")
print(f"Hindi danda-normalised WER: {hi_wer['wer']:.2%}")

Reference : the cat sat on the mat
Hypothesis: the cat sit on a mat
WER: 33.33% (S=2, D=0, I=0)
CER: 23.53%
Hindi danda-normalised WER: 0.00%


## Part 1 Results: Prisma ASR (Real Human Speech + Synthetic Smoke)

In [4]:
with open('results/asr_results.json', 'r', encoding='utf-8') as f:
    asr_res = json.load(f)

for suite_name, suite in asr_res['suites'].items():
    ov = suite['overall']
    print('=' * 74)
    print(f"SUITE: {suite_name} (n={suite['metadata']['successful_clips']}/{suite['metadata']['total_clips']})")
    print(f"Verbatim WER: {ov['wer']:.2%}  |  Format-Normalised WER: {ov['formatted_wer']:.2%}")
    print(f"Verbatim CER: {ov['cer']:.2%}  |  Format-Normalised CER: {ov['formatted_cer']:.2%}")
    print(f"Latency: p50={ov['p50_latency_ms']:.0f}ms, p95={ov['p95_latency_ms']:.0f}ms, mean={ov['avg_latency_ms']:.0f}ms, retries={ov['total_retries']}")
    print('-' * 74)
    print(f"{'Category':<24} | {'Verb WER':>8} | {'Fmt WER':>8} | {'p50 Lat':>8} | {'Clips':>5}")
    for cat, d in suite['by_category'].items():
        print(f"{cat:<24} | {d['aggregate_wer']*100:>7.2f}% | {d['formatted_wer']*100:>7.2f}% | {d['p50_latency_ms']:>6.0f}ms | {d['num_clips']:>5}")
    print()

SUITE: real_human (n=22/22)
Verbatim WER: 11.07%  |  Format-Normalised WER: 10.40%
Verbatim CER: 9.95%  |  Format-Normalised CER: 9.02%
Latency: p50=388ms, p95=634ms, mean=394ms, retries=0
--------------------------------------------------------------------------
Category                 | Verb WER |  Fmt WER |  p50 Lat | Clips
real_indian_english      |    6.25% |    2.08% |    296ms |     5
real_hindi               |   14.63% |   14.63% |    516ms |     5
real_regional_indic      |   14.55% |   14.55% |    406ms |     8
real_noisy_audio         |    2.78% |    2.78% |    305ms |     2
real_telephony           |    8.33% |    8.33% |    316ms |     2

SUITE: synthetic_smoke (n=30/30)
Verbatim WER: 20.00%  |  Format-Normalised WER: 0.37%
Verbatim CER: 18.62%  |  Format-Normalised CER: 0.49%
Latency: p50=340ms, p95=490ms, mean=344ms, retries=0
--------------------------------------------------------------------------
Category                 | Verb WER |  Fmt WER |  p50 Lat | Clips
clea

## Part 2 Results: Timbre TTS (Component Metrics Across Repeats)

In [5]:
with open('results/tts_results.json', 'r', encoding='utf-8') as f:
    tts_res = json.load(f)

ov = tts_res['overall']
meta = tts_res['metadata']
print('=' * 78)
print(f"TIMBRE TTS BENCHMARK (n={meta['successful']}/{meta['total_sentences']}, repeats={meta['repeats_per_sentence']})")
print(f"Full-response HTTP Latency: p50={ov['p50_latency_ms']:.0f}ms, p95={ov['p95_latency_ms']:.0f}ms, mean={ov['avg_latency_ms']:.0f}ms, retries={ov['total_retries']}")
print(f"RTF: p50={ov['p50_rtf']:.4f}, p95={ov['p95_rtf']:.4f}, mean={ov['avg_rtf']:.4f}  |  25ms Silence Ratio: {ov['avg_silence_ratio']:.3f}")
print(f"Round-trip Prisma WER: verbatim={ov['avg_roundtrip_verbatim_wer']:.2%}, formatted={ov['avg_roundtrip_formatted_wer']:.2%}")
print('=' * 78)
print(f"{'Category':<24} | {'p50 Lat':>7} | {'p95 Lat':>7} | {'p50 RTF':>7} | {'Sil':>5} | {'RT Fmt WER':>10}")
for cat, d in tts_res['by_category'].items():
    rt = f"{d['avg_roundtrip_formatted_wer']*100:.1f}%" if d['avg_roundtrip_formatted_wer'] is not None else 'n/a'
    print(f"{cat:<24} | {d['p50_latency_ms']:>5.0f}ms | {d['p95_latency_ms']:>5.0f}ms | {d['p50_rtf']:>7.4f} | {d['avg_silence_ratio']:>5.3f} | {rt:>10}")

TIMBRE TTS BENCHMARK (n=28/29, repeats=3)
Full-response HTTP Latency: p50=852ms, p95=1321ms, mean=908ms, retries=0
RTF: p50=0.1713, p95=0.2325, mean=0.1737  |  25ms Silence Ratio: 0.242
Round-trip Prisma WER: verbatim=25.71%, formatted=6.21%
Category                 | p50 Lat | p95 Lat | p50 RTF |   Sil | RT Fmt WER
numbers                  |   859ms |   954ms |  0.1852 | 0.236 |       0.0%
abbreviations            |   899ms |   980ms |  0.1578 | 0.219 |       0.0%
code_switching           |   725ms |   849ms |  0.1618 | 0.224 |      11.1%
long_form                |  1691ms |  1961ms |  0.0822 | 0.259 |       3.9%
tricky_pronunciation     |   727ms |   936ms |  0.1889 | 0.197 |       9.2%
punctuation_and_prosody  |   946ms |  1086ms |  0.1742 | 0.386 |       0.0%
special_formats          |  1073ms |  1179ms |  0.1625 | 0.228 |      29.2%
regional_languages       |   854ms |   896ms |  0.1705 | 0.246 |       2.8%
edge_cases               |   736ms |   916ms |  0.1888 | 0.260 |       0.0

## Live Interactive Check (Native Voice + Prisma Round-Trip)

In [6]:
client = GnaniClient()
test_text = 'Aapka OTP 4523 hai, please isse kisi ke saath share mat kijiye.'
output_audio = 'data/tts/output/demo_sample.wav'

tts_out = client.synthesize(text=test_text, language='hi-en', output_path=output_audio)
print(f"Synthesized with voice={tts_out['voice']} in {tts_out['latency_ms']} ms (retries={tts_out['retries']})")

asr_out = client.transcribe(output_audio, language='hi-IN,en-IN')
print(f"Transcribed in {asr_out['latency_ms']} ms: {asr_out['transcript']}")

Synthesized with voice=Poorvi in 876.36 ms (retries=0)
Transcribed in 397.62 ms: आपका ओटीपी 4523 है प्लीज इसे किसी के साथ शेयर मैट कीजिए।
